# Module ❶: Tăng trưởng & Tài chính - Descriptive Analysis
**Task 2.1: Revenue YoY Trend & Seasonality**
*Owner: Lê Bảo Khánh*

Mục tiêu:
- Phân tích xu hướng doanh thu qua các năm (2012-2022).
- Tính toán tốc độ tăng trưởng (YoY Growth Rate) và CAGR.
- Phát hiện tính mùa vụ (Seasonality) theo tháng/quý.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import sys
import os

# Add src to path if needed
sys.path.append(os.path.abspath(os.path.join('..', 'src')))
from data_loader import DataLoader

# Set random state and plot style
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

In [ ]:
# 1. Load data: sales, order_items, products
loader = DataLoader()

# Load necessary tables
sales = loader.load("sales")
order_items = loader.load("order_items")
products = loader.load("products")

print("Sales data shape:", sales.shape)
print("Date range:", sales["Date"].min(), "to", sales["Date"].max())
print("\n--- 5 DÒNG ĐẦU & CUỐI (Preview Data) ---")
print(sales)
print("\n--- THỐNG KÊ TỔNG QUAN TOÀN BỘ GIAI ĐOẠN 2012-2022 ---")
print(sales.describe())

## 1. Xu hướng Doanh thu (Revenue YoY Trend)
Phân tích tổng doanh thu theo năm và tốc độ tăng trưởng (Growth Rate).

In [ ]:
# Thêm cột Year và Month
sales = sales.with_columns([
    pl.col("Date").dt.year().alias("Year"),
    pl.col("Date").dt.month().alias("Month"),
    pl.col("Date").dt.quarter().alias("Quarter")
])

# Group by Year: Tính Total Revenue và Total COGS
yearly_sales = sales.group_by("Year").agg([
    pl.col("Revenue").sum().alias("Total_Revenue"),
    pl.col("COGS").sum().alias("Total_COGS")
]).sort("Year")

# Lấy năm gốc và doanh thu gốc
base_year = yearly_sales["Year"][0]
base_revenue = yearly_sales["Total_Revenue"][0]

# Tính YoY Growth Rate và CAGR từ năm gốc cho từng năm
yearly_sales = yearly_sales.with_columns([
    (pl.col("Total_Revenue").diff() / pl.col("Total_Revenue").shift(1) * 100).alias("YoY_Growth_Rate_%"),
    pl.when(pl.col("Year") == base_year)
      .then(None)
      .otherwise(
          (((pl.col("Total_Revenue") / base_revenue) ** (1.0 / (pl.col("Year") - base_year))) - 1) * 100
      ).alias("CAGR_from_Base_Year_%")
])

# Hiển thị tất cả các năm (2012-2022) không bị ẩn dòng
with pl.Config(tbl_rows=20):
    print(yearly_sales)

In [ ]:
# Vẽ Line Chart YoY
fig, ax1 = plt.subplots(figsize=(12, 6))

years = yearly_sales["Year"].to_list()
revenue = yearly_sales["Total_Revenue"].to_list()
growth = yearly_sales["YoY_Growth_Rate_%"].to_list()

color = 'tab:blue'
ax1.set_xlabel('Year', fontweight='bold')
ax1.set_ylabel('Total Revenue ($)', color=color, fontweight='bold')
line1 = ax1.plot(years, revenue, marker='o', linewidth=2.5, color=color, label='Revenue')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xticks(years)

# Format y-axis to millions
formatter = plt.FuncFormatter(lambda x, pos: f'${x*1e-6:.1f}M')
ax1.yaxis.set_major_formatter(formatter)

# Dual axis for Growth Rate
ax2 = ax1.twinx()  
color2 = 'tab:red'
ax2.set_ylabel('YoY Growth Rate (%)', color=color2, fontweight='bold')
line2 = ax2.plot(years, growth, marker='s', linestyle='--', color=color2, label='Growth Rate (%)')
ax2.tick_params(axis='y', labelcolor=color2)

# Annotate anomalies/key points
for i, txt in enumerate(growth):
    if txt is not None and not np.isnan(txt):
        ax2.annotate(f"{txt:.1f}%", (years[i], growth[i]), textcoords="offset points", xytext=(0,10), ha='center', color=color2)

plt.title('Revenue YoY Trend & Growth Rate (2012 - 2022)', fontsize=16, fontweight='bold')
fig.tight_layout()

# Legend
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left')

plt.show()

In [ ]:
# Tính CAGR (Compound Annual Growth Rate) toàn giai đoạn
# CAGR = (Ending Value / Beginning Value) ^ (1 / Number of Years) - 1

start_year = yearly_sales["Year"][0]
end_year = yearly_sales["Year"][-1]
num_years = end_year - start_year

start_revenue = yearly_sales.filter(pl.col("Year") == start_year)["Total_Revenue"][0]
end_revenue = yearly_sales.filter(pl.col("Year") == end_year)["Total_Revenue"][0]

cagr = ((end_revenue / start_revenue) ** (1 / num_years)) - 1
print(f"Bắt đầu ({start_year}): ${start_revenue:,.2f}")
print(f"Kết thúc ({end_year}): ${end_revenue:,.2f}")
print(f"CAGR ({start_year}-{end_year}): {cagr * 100:.2f}%")

## 2. Tính Mùa Vụ (Seasonality)
Phân tích theo tháng để phát hiện các mẫu (patterns) lặp lại hàng năm.

In [ ]:
# Group by Year and Month
monthly_sales = sales.group_by(["Year", "Month"]).agg(
    pl.col("Revenue").sum().alias("Monthly_Revenue")
).sort(["Year", "Month"])

# Pivot data for heatmap
pivot_df = monthly_sales.pivot(values="Monthly_Revenue", index="Year", on="Month")
# Fill nulls with 0 or drop incomplete years
pivot_df = pivot_df.fill_null(0)

# Convert to pandas for easier seaborn heatmap plotting
pd_pivot = pivot_df.to_pandas().set_index("Year")
# Sort columns (months) in case they are out of order
pd_pivot = pd_pivot[sorted([c for c in pd_pivot.columns if str(c).isdigit()])]

plt.figure(figsize=(14, 6))
sns.heatmap(pd_pivot / 1e6, cmap="YlGnBu", annot=True, fmt=".1f", cbar_kws={'label': 'Revenue ($ Millions)'})
plt.title('Monthly Revenue Heatmap (in $ Millions)', fontsize=16, fontweight='bold')
plt.xlabel('Month', fontweight='bold')
plt.ylabel('Year', fontweight='bold')
plt.show()

# Trung bình doanh thu theo tháng (box plot)
plt.figure(figsize=(12, 6))
sns.boxplot(data=monthly_sales.to_pandas(), x="Month", y="Monthly_Revenue", palette="viridis")
plt.title('Revenue Distribution by Month (Seasonality Check)', fontsize=16, fontweight='bold')
plt.xlabel('Month', fontweight='bold')
plt.ylabel('Revenue ($)', fontweight='bold')

formatter = plt.FuncFormatter(lambda x, pos: f'${x*1e-6:.1f}M')
plt.gca().yaxis.set_major_formatter(formatter)
plt.show()

## 📝 Narrative - Business Insights
Dựa vào phân tích trên:
1. **Xu hướng tăng trưởng (YoY Trend)**:
   - Doanh nghiệp có chỉ số tăng trưởng (Growth Rate) ổn định hay biến động?
   - Tính toán CAGR cho thấy bức tranh dài hạn (CAGR ~ X%). 
   - Điểm đáng chú ý: Năm nào có sự sụt giảm hoặc tăng đột biến? (Cần investigate thêm trong phần Diagnostic).
   
2. **Tính mùa vụ (Seasonality)**:
   - Doanh thu cao nhất thường rơi vào những tháng nào? (Ví dụ: Tháng 11-12 do mùa lễ hội/khuyến mãi).
   - Có tháng nào luôn ở mức thấp không? 
   - Thông tin này hữu ích để chuẩn bị inventory (Module 4) và dồn ngân sách marketing (Module 5).

## 3. Margin & AOV Trend (Task 2.2)
Phân tích Gross Margin, AOV (Average Order Value) và RPU (Revenue Per User).

In [ ]:
# Load orders data since it has customer_id and order_date
orders = loader.load("orders")

# Tính Revenue và COGS từ order_items và products
# 1. Join order_items với products để lấy cogs
order_details = order_items.join(products.select(["product_id", "cogs"]), on="product_id", how="left")

# 2. Tính Revenue = (quantity * unit_price) - discount_amount
#    Tính Total COGS = quantity * cogs
order_details = order_details.with_columns([
    ((pl.col("quantity") * pl.col("unit_price")) - pl.col("discount_amount")).alias("item_revenue"),
    (pl.col("quantity") * pl.col("cogs")).alias("item_cogs")
])

# 3. Aggregate theo order_id
order_aggs = order_details.group_by("order_id").agg([
    pl.col("item_revenue").sum().alias("order_revenue"),
    pl.col("item_cogs").sum().alias("order_cogs")
])

# 4. Join với orders để có order_date và customer_id
orders_full = orders.join(order_aggs, on="order_id", how="inner")

# Thêm cột Year
orders_full = orders_full.with_columns(pl.col("order_date").dt.year().alias("Year"))

# 5. Tính các metric theo Year
yearly_metrics = orders_full.group_by("Year").agg([
    pl.col("order_revenue").sum().alias("Total_Revenue"),
    pl.col("order_cogs").sum().alias("Total_COGS"),
    pl.col("order_id").n_unique().alias("Num_Orders"),
    pl.col("customer_id").n_unique().alias("Num_Customers")
]).sort("Year")

# 6. Tính Gross Margin, AOV, RPU
yearly_metrics = yearly_metrics.with_columns([
    ((pl.col("Total_Revenue") - pl.col("Total_COGS")) / pl.col("Total_Revenue") * 100).alias("Gross_Margin_%"),
    (pl.col("Total_Revenue") / pl.col("Num_Orders")).alias("AOV"),
    (pl.col("Total_Revenue") / pl.col("Num_Customers")).alias("RPU")
])

print(yearly_metrics.select(["Year", "Gross_Margin_%", "AOV", "RPU"]))

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 12), sharex=True)

years = yearly_metrics["Year"].to_list()
margins = yearly_metrics["Gross_Margin_%"].to_list()
aovs = yearly_metrics["AOV"].to_list()
rpus = yearly_metrics["RPU"].to_list()

# 1. Gross Margin
ax1.plot(years, margins, marker='o', color='tab:orange', linewidth=2.5)
ax1.set_ylabel('Gross Margin (%)', fontweight='bold', color='tab:orange')
ax1.set_title('Gross Margin Trend (2012-2022)', fontweight='bold')
ax1.tick_params(axis='y', labelcolor='tab:orange')

# Annotate anomaly if any (e.g., minimum margin)
min_margin_idx = np.argmin(margins)
ax1.annotate(f'Min: {margins[min_margin_idx]:.1f}%', 
             (years[min_margin_idx], margins[min_margin_idx]),
             textcoords="offset points", xytext=(0,-15), ha='center', color='red', fontweight='bold')

# 2. AOV
ax2.plot(years, aovs, marker='s', color='tab:green', linewidth=2.5)
ax2.set_ylabel('AOV ($)', fontweight='bold', color='tab:green')
ax2.set_title('Average Order Value (AOV) Trend', fontweight='bold')
ax2.tick_params(axis='y', labelcolor='tab:green')

# 3. RPU
ax3.plot(years, rpus, marker='^', color='tab:purple', linewidth=2.5)
ax3.set_ylabel('RPU ($)', fontweight='bold', color='tab:purple')
ax3.set_title('Revenue Per User (RPU) Trend', fontweight='bold')
ax3.tick_params(axis='y', labelcolor='tab:purple')
ax3.set_xlabel('Year', fontweight='bold')
ax3.set_xticks(years)

fig.tight_layout()
plt.show()

## 📝 Narrative - Margin & AOV/RPU Insights
Dựa vào các biểu đồ trên:
1. **Gross Margin**: Biên lợi nhuận gộp đang có xu hướng như thế nào? Chú ý điểm Min margin đỏ trên biểu đồ. Nếu có hiện tượng *Margin Squeeze* (tức là margin giảm liên tục hoặc tụt sâu ở những năm gần đây), chúng ta cần tìm hiểu nguyên nhân (COGS tăng hay giá bán giảm) trong phần Diagnostic.
2. **AOV (Average Order Value)**: Giá trị đơn hàng trung bình tăng hay giảm? Nếu AOV giảm, có thể do khách mua các mặt hàng rẻ hơn hoặc discount nhiều hơn.
3. **RPU (Revenue Per User)**: Cho thấy một khách hàng trung bình mang lại bao nhiêu doanh thu trong năm. RPU đi ngang hay giảm báo hiệu vấn đề về mức độ trung thành hoặc sức mua của KH.

---
## 3. Margin & AOV Trend (Task 2.2)
Mục tiêu:
- Tính toán Gross Margin, AOV (Average Order Value) và RPU (Revenue Per User).
- Phát hiện các điểm bất thường (anomaly) theo thời gian.

In [ ]:
# Load orders data
orders = loader.load("orders")

# Thêm cột Year vào orders
orders = orders.with_columns([
    pl.col("order_date").dt.year().alias("Year")
])

# Tính số đơn hàng và số lượng unique users theo năm
yearly_orders = orders.group_by("Year").agg([
    pl.col("order_id").n_unique().alias("Total_Orders"),
    pl.col("customer_id").n_unique().alias("Unique_Users")
]).sort("Year")

# Tính tổng COGS theo năm từ sales data để tính Gross Margin
yearly_margin = sales.group_by("Year").agg([
    pl.col("Revenue").sum().alias("Total_Revenue"),
    pl.col("COGS").sum().alias("Total_COGS")
]).sort("Year")

# Join các bảng lại với nhau
metrics = yearly_margin.join(yearly_orders, on="Year", how="left")

# Tính các chỉ số
metrics = metrics.with_columns([
    ((pl.col("Total_Revenue") - pl.col("Total_COGS")) / pl.col("Total_Revenue") * 100).alias("Gross_Margin_%"),
    (pl.col("Total_Revenue") / pl.col("Total_Orders")).alias("AOV"),
    (pl.col("Total_Revenue") / pl.col("Unique_Users")).alias("RPU")
])

with pl.Config(tbl_rows=20):
    print(metrics)

In [ ]:
# Vẽ biểu đồ Multi-line chart so sánh xu hướng các chỉ số
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

years = metrics["Year"].to_list()
margin = metrics["Gross_Margin_%"].to_list()
aov = metrics["AOV"].to_list()
rpu = metrics["RPU"].to_list()

# 1. Gross Margin Trend
axes[0].plot(years, margin, marker="o", color="tab:green", linewidth=2)
axes[0].set_ylabel("Gross Margin (%)", fontweight="bold")
axes[0].set_title("Gross Margin Trend (2012-2022)", fontweight="bold")
for i, txt in enumerate(margin):
    if txt is not None and not np.isnan(txt):
        axes[0].annotate(f"{txt:.1f}%", (years[i], margin[i]), textcoords="offset points", xytext=(0,10), ha="center")

# 2. AOV Trend
axes[1].plot(years, aov, marker="s", color="tab:orange", linewidth=2)
axes[1].set_ylabel("AOV ($)", fontweight="bold")
axes[1].set_title("Average Order Value (AOV) Trend", fontweight="bold")
for i, txt in enumerate(aov):
    if txt is not None and not np.isnan(txt):
        axes[1].annotate(f"${txt:.1f}", (years[i], aov[i]), textcoords="offset points", xytext=(0,10), ha="center")

# 3. RPU Trend
axes[2].plot(years, rpu, marker="^", color="tab:purple", linewidth=2)
axes[2].set_ylabel("RPU ($)", fontweight="bold")
axes[2].set_xlabel("Year", fontweight="bold")
axes[2].set_title("Revenue Per User (RPU) Trend", fontweight="bold")
axes[2].set_xticks(years)
for i, txt in enumerate(rpu):
    if txt is not None and not np.isnan(txt):
        axes[2].annotate(f"${txt:.1f}", (years[i], rpu[i]), textcoords="offset points", xytext=(0,10), ha="center")

plt.tight_layout()
plt.show()

## 📝 Narrative - Margin & AOV Insights
1. **Gross Margin**:
   - Tỷ suất lợi nhuận gộp có duy trì ổn định hay đang bị thu hẹp (Margin Squeeze)?
   - Liệu sự thay đổi của COGS (do lạm phát, chi phí nguyên vật liệu) có ảnh hưởng mạnh đến biên lợi nhuận hay không?
2. **AOV & RPU**:
   - Giá trị đơn hàng trung bình (AOV) và doanh thu trên mỗi người dùng (RPU) đang có xu hướng tăng hay giảm?
   - Liệu công ty đang thu hút thêm người dùng mới với giá trị mua thấp hơn, hay giữ chân người dùng cũ với mức chi tiêu cao hơn?
   - Các điểm gãy (anomaly) trong đồ thị có thể là dấu hiệu của các chiến dịch khuyến mãi lớn hoặc sự thay đổi trong chiến lược giá.